# Reshaping — `long`, `wide` & `crosstab`

`long` melts numeric columns to rows. `wide` pivots a column's values into headers. `crosstab` computes frequency matrices and contingency tables across categorical dimensions.

| Method | Key params | Purpose |
|---|---|---|
| `.pt.long(c='variable', v='value')` | `c`, `v` | wide → long (melt) |
| `.pt.wide(..., a=None, dropna=True)` | `c`, `v`, `a`, `dropna` | long → wide (pivot); `dropna` only with `a` / pivot_table |
| `.pt.crosstab(index, columns, ...)` | `index`, `columns`, `values`, `aggfunc`, `normalize`, `margins` | frequency cross-tabulation & contingency matrix |

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
healthexp = pt.sample_data['healthexp']
flights = pt.sample_data['flights']
tips = pt.sample_data['tips']

In [2]:
penguins

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


## `long` — melt all numeric columns to rows

In [3]:
# Melt all numeric columns to rows — default col/value names
(
    penguins
    .pt.long()
    .sample(10)
)


,species,island,sex,variable,value
611,Gentoo,Biscoe,Male,bill_depth_mm,15.9
381,Adelie,Dream,Female,bill_depth_mm,18.5
891,Chinstrap,Dream,Male,flipper_length_mm,201.0
1000,Gentoo,Biscoe,Female,flipper_length_mm,212.0
831,Adelie,Dream,Male,flipper_length_mm,190.0
211,Chinstrap,Dream,Female,bill_length_mm,45.6
954,Gentoo,Biscoe,Female,flipper_length_mm,210.0
724,Adelie,Dream,Male,flipper_length_mm,190.0
515,Chinstrap,Dream,Male,bill_depth_mm,18.2
593,Gentoo,Biscoe,Male,bill_depth_mm,15.3


In [4]:
# Custom column names for the variable and value columns
(
    penguins
    .pt.long(c='metric', v='reading')
    .sample(10)
)


,species,island,sex,metric,reading
1257,Gentoo,Biscoe,Female,body_mass_g,4550.0
925,Gentoo,Biscoe,Male,flipper_length_mm,221.0
1214,Chinstrap,Dream,Female,body_mass_g,3200.0
1019,Gentoo,Biscoe,Male,flipper_length_mm,229.0
173,Chinstrap,Dream,Male,bill_length_mm,48.5
1000,Gentoo,Biscoe,Female,flipper_length_mm,212.0
458,Adelie,Biscoe,Female,bill_depth_mm,20.7
204,Chinstrap,Dream,Female,bill_length_mm,45.7
28,Adelie,Biscoe,Female,bill_length_mm,37.9
215,Chinstrap,Dream,Male,bill_length_mm,55.8


In [5]:
# Only non numeric columns survive as id columns — numerics all get melted
# select narrows which numerics get melted
(penguins
 .pt.select('species', 'island', 'bill_length_mm', 'bill_depth_mm')
 .pt.long(c='metric', v='reading')
 .sample(12)
)


,species,island,metric,reading
144,Adelie,Dream,bill_length_mm,37.3
174,Chinstrap,Dream,bill_length_mm,43.2
574,Gentoo,Biscoe,bill_depth_mm,13.7
68,Adelie,Torgersen,bill_length_mm,35.9
122,Adelie,Torgersen,bill_length_mm,40.2
70,Adelie,Torgersen,bill_length_mm,33.5
192,Chinstrap,Dream,bill_length_mm,49.0
172,Chinstrap,Dream,bill_length_mm,42.4
206,Chinstrap,Dream,bill_length_mm,42.5
392,Adelie,Dream,bill_depth_mm,17.9


## `wide` — pivot a column's values into headers

In [6]:
# flights has one row per year+month — no aggfunc needed
(
    flights
    .pt.wide(c='month', v='passengers')
    .head()
)


,year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,1949,112,118,132,129,121,135,148,148,136,119,104,118
1,1950,115,126,141,135,125,149,170,170,158,133,114,140
2,1951,145,150,178,163,172,178,199,199,184,162,146,166
3,1952,171,180,193,181,183,218,230,242,209,191,172,194
4,1953,196,196,236,235,229,243,264,272,237,211,180,201


In [7]:
# Select only the columns needed so island becomes the clean index
(penguins
 .pt.select('island', 'species', 'body_mass_g')
 .pt.wide(c='species', v='body_mass_g', a='mean')  # dropna=True by default
)


,island,Adelie,Chinstrap,Gentoo
0,Biscoe,3709.659091,NaN,5076.01626
1,Dream,3688.392857,3733.088235,NaN
2,Torgersen,3706.372549,NaN,NaN


`dropna` only applies with `aggfunc` / `pivot_table`. Country `cn` has only a missing balance, so after the pivot that column is all-NA.

In [8]:
import pandas as pd
tiny = pd.DataFrame({
    'id': ['a', 'b', 'b'],
    'country': ['sg', 'sg', 'cn'],
    'balance': [10.0, 20.0, None],
})
tiny

,id,country,balance
0,a,sg,10.0
1,b,sg,20.0
2,b,cn,NaN


In [9]:
tiny.pt.wide(c='country', v='balance', a='mean')  # dropna=True: all-NA column cn dropped

,id,sg
0,a,10.0
1,b,20.0


In [10]:
tiny.pt.wide(c='country', v='balance', a='mean', dropna=False)  # keep cn

,id,cn,sg
0,a,NaN,10.0
1,b,NaN,20.0


## Round-trip: `long` → transform → `wide`
A common pattern: go long to apply uniform operations across all metrics, then come back wide.

In [11]:
# Select only needed columns → long → wide: yields mean bill metrics per island
(penguins
 .pt.select('island', 'bill_length_mm', 'bill_depth_mm')
 .pt.long(c='metric', v='reading')
 .pt.wide(c='metric', v='reading', a='mean')
)


,island,bill_depth_mm,bill_length_mm
0,Biscoe,15.874850,45.257485
1,Dream,18.344355,44.167742
2,Torgersen,18.429412,38.950980


---

# `crosstab` — Cross-Tabulation & Frequency Matrices

`pt.crosstab(df, index, columns, ...)` and `df.pt.crosstab(...)` compute multi-way contingency tables and frequency matrices.

| Parameter | Type | Description |
|---|---|---|
| `index` | `str | list[str]` | Column(s) to group by on the rows |
| `columns` | `str` | Column to group by across the columns |
| `values` | `str | None` | Optional numeric column to aggregate |
| `aggfunc` | `str | callable | None` | Aggregation function (required if `values` is given) |
| `normalize` | `bool | str` | `'index'` (row %), `'columns'` (col %), `'all'` / `True` (grand total %) |
| `margins` | `bool` | Add row/column sub-totals and grand totals (`margins=True`) |
| `margins_name` | `str` | Label for marginal row/columns (default `'All'`) |
| `dropna` | `bool` | Drop columns/rows with only missing values (default `True`) |

Bracket notation `[col]` is supported for columns with spaces across `index`, `columns`, and `values`.

In [12]:
# 1. Basic 2-way frequency matrix (counts)
tips.pt.crosstab('smoker', 'day')

day,Thur,Fri,Sat,Sun
smoker,,,,
Yes,17,15,42,19
No,45,4,45,57


### Marginal totals (`margins=True`, `margins_name=`)
Compute row sums, column sums, and grand totals with a custom label:

In [13]:
# 2. Marginal totals with custom label
tips.pt.crosstab('smoker', 'day', margins=True, margins_name='Total')

day,Thur,Fri,Sat,Sun,Total
smoker,,,,,
Yes,17,15,42,19,93
No,45,4,45,57,151
Total,62,19,87,76,244


### Proportions & Normalization (`normalize=`)
Normalize frequencies across rows (`normalize='index'`), columns (`normalize='columns'`), or grand total (`normalize='all'`):

In [14]:
# 3a. Row proportions (sum to 1.0 per row)
tips.pt.crosstab('smoker', 'day', normalize='index').round(3)

day,Thur,Fri,Sat,Sun
smoker,,,,
Yes,0.183,0.161,0.452,0.204
No,0.298,0.026,0.298,0.377


In [15]:
# 3b. Grand total proportions (sum to 1.0 across entire matrix)
tips.pt.crosstab('smoker', 'day', normalize='all').round(3)

day,Thur,Fri,Sat,Sun
smoker,,,,
Yes,0.070,0.061,0.172,0.078
No,0.184,0.016,0.184,0.234


### Aggregating values (`values=`, `aggfunc=`)
Instead of counting frequencies, compute summary statistics on a numeric column (e.g. mean `total_bill`):

In [16]:
# 4. Summary statistic: mean total bill by smoker and day
tips.pt.crosstab('smoker', 'day', values='total_bill', aggfunc='mean').round(2)

day,Thur,Fri,Sat,Sun
smoker,,,,
Yes,19.19,16.81,21.28,24.12
No,17.11,18.42,19.66,20.51


### Multi-Column Index
Cross-tabulate across multiple row dimensions (e.g., `species` and `sex` across islands):

In [17]:
# 5. Multi-column index
penguins.pt.crosstab(index=['species', 'sex'], columns='island')

island            Biscoe  Dream  Torgersen
species   sex                             
Adelie    Female      22     27         24
          Male        22     28         23
Chinstrap Female       0     34          0
          Male         0     34          0
Gentoo    Female      58      0          0
          Male        61      0          0

### Columns with spaces (`[...]` bracket notation)
Columns with spaces are fully supported in `index`, `columns`, and `values` using brackets:

In [18]:
# 6. Spaced columns using bracket syntax
tips_spaced = tips.rename(columns={
    'smoker': 'smoker status',
    'day': 'day of week',
    'total_bill': 'total bill',
})

tips_spaced.pt.crosstab(
    index='[smoker status]',
    columns='[day of week]',
    values='[total bill]',
    aggfunc='mean',
).round(2)

day of week,Thur,Fri,Sat,Sun
smoker status,,,,
Yes,19.19,16.81,21.28,24.12
No,17.11,18.42,19.66,20.51


### Pipeline chaining
Chain with `.pt.qry()`, `.rename()`, and other operations seamlessly:

In [19]:
# 7. Pipeline chaining: filter weekend dinners with parties >= 2, then crosstab
(
    tips
    .pt.qry(day=['Sat', 'Sun'], time='Dinner', size='>= 2')
    .pt.crosstab('smoker', 'day', values='total_bill', aggfunc='mean')
    .round(2)
)

day,Sat,Sun
smoker,,
Yes,21.72,24.12
No,19.94,20.51
